# Notebook 3 — Combined LangChain + LangGraph QA Example
## Defect Report Analyser & Recommendation Agent

### What this notebook does
You paste a defect description. The agent:
1. **Classifies** the defect — severity, type, affected area
2. **Routes** it — Critical defects go to an urgent handler, others to standard
3. **Recommends** — what to test next, what the root cause might be
4. **Summarises** — a one-paragraph defect analysis report

### What you will practise
- Everything from Notebook 1 (LangChain: prompts, chains, parsers)
- Everything from Notebook 2 (LangGraph: state, nodes, conditional edges)
- Putting it all together into a real QA workflow

> This is the same pattern used in the 3 QA agent notebooks you already have

---
## Step 1 — Install

In [1]:
%pip install -q langchain langchain-ollama langgraph

Note: you may need to restart the kernel to use updated packages.


---
## Step 2 — All Imports

In [1]:
# Standard library
from typing import TypedDict

# LangGraph
from langgraph.graph import StateGraph, END

# LangChain
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown

print('All imports OK')

All imports OK


---
## Step 3 — Create the LLM

In [2]:
# The LLM is created once and shared across all nodes
# All nodes in the graph use this same llm object
llm = ChatOllama(
    model='llama3.2',
    base_url='http://localhost:11434',
    temperature=0.3,
)
print('LLM ready:', llm.model)

LLM ready: llama3.2


---
## Step 4 — Define the State
Every field is one piece of data passed between nodes.

In [3]:
class DefectState(TypedDict):
    defect_description: str   # input — the raw defect report
    classification:     str   # Node 1 output — severity, type, area
    severity_level:     str   # extracted from classification — 'critical' or 'standard'
    recommendation:     str   # Node 3 or 4 output — what to do next
    final_report:       str   # Node 5 output — the complete analysis

print('State defined with fields:')
for field, ftype in DefectState.__annotations__.items():
    print(f'  {field}: {ftype.__name__}')

State defined with fields:
  defect_description: str
  classification: str
  severity_level: str
  recommendation: str
  final_report: str


---
## Step 5 — Build All Chains (LangChain part)
Each chain is: prompt | llm | parser

In [4]:
# ── Chain 1: Classifier ──────────────────────────────────────────
# Reads the defect and classifies it
classify_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a QA defect analyst for banking and insurance systems. '
     'Classify the defect and respond in this exact format:\n'
     'SEVERITY: [Critical/High/Medium/Low]\n'
     'TYPE: [Functional/Performance/Security/UI/Data]\n'
     'AFFECTED AREA: [e.g. Fund Transfer, Login, Claims, Reports]\n'
     'SUMMARY: [one sentence describing the defect]'),
    ('human', 'Defect Report:\n{defect}')
])
classify_chain = classify_prompt | llm | StrOutputParser()

print('Chain 1 (Classifier) ready')

Chain 1 (Classifier) ready


In [5]:
# ── Chain 2: Critical Handler ────────────────────────────────────
# Only runs for Critical/High severity defects
critical_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a senior QA lead handling a CRITICAL defect in a financial system. '
     'Provide:\n'
     '1. Immediate actions (what to do in the next 2 hours)\n'
     '2. Likely root cause\n'
     '3. Regression areas to retest\n'
     '4. Rollback recommendation (yes/no and why)'),
    ('human',
     'Classification:\n{classification}\n\n'
     'Original Defect:\n{defect}')
])
critical_chain = critical_prompt | llm | StrOutputParser()

print('Chain 2 (Critical Handler) ready')

Chain 2 (Critical Handler) ready


In [6]:
# ── Chain 3: Standard Handler ────────────────────────────────────
# Runs for Medium/Low severity defects
standard_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a QA analyst handling a standard defect. Provide:\n'
     '1. Suggested fix approach\n'
     '2. Test cases to verify the fix\n'
     '3. Related areas to do a quick sanity check on'),
    ('human',
     'Classification:\n{classification}\n\n'
     'Original Defect:\n{defect}')
])
standard_chain = standard_prompt | llm | StrOutputParser()

print('Chain 3 (Standard Handler) ready')

Chain 3 (Standard Handler) ready


In [7]:
# ── Chain 4: Report Writer ───────────────────────────────────────
# Runs last — always — combines everything into a final report
report_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a QA report writer. Write a professional one-paragraph '
     'defect analysis report suitable for a manager. '
     'Include severity, impact, and recommended next steps. '
     'Be concise and factual.'),
    ('human',
     'Defect: {defect}\n\n'
     'Classification: {classification}\n\n'
     'Recommendation: {recommendation}')
])
report_chain = report_prompt | llm | StrOutputParser()

print('Chain 4 (Report Writer) ready')

Chain 4 (Report Writer) ready


---
## Step 6 — Define All Nodes (LangGraph part)
Each node calls one chain.

In [8]:
# Node 1 — Classifier
# Reads: defect_description
# Writes: classification, severity_level
def node_classifier(state: DefectState) -> DefectState:
    print('[Node 1 - Classifier] Classifying defect...')

    classification = classify_chain.invoke({
        'defect': state['defect_description']
    })

    # Extract severity level for routing
    # We check if 'critical' or 'high' appears in the classification text
    classification_lower = classification.lower()
    if 'critical' in classification_lower or 'high' in classification_lower:
        severity_level = 'critical'
    else:
        severity_level = 'standard'

    print(f'  Severity level determined: {severity_level}')

    return {
        **state,
        'classification':  classification,
        'severity_level':  severity_level
    }

print('Node 1 defined')

Node 1 defined


In [9]:
# Node 2 — Critical Handler (only runs for critical/high defects)
# Reads: classification, defect_description
# Writes: recommendation
def node_critical_handler(state: DefectState) -> DefectState:
    print('[Node 2 - Critical Handler] Processing critical defect...')

    recommendation = critical_chain.invoke({
        'classification': state['classification'],
        'defect':         state['defect_description']
    })

    return {**state, 'recommendation': recommendation}


# Node 3 — Standard Handler (only runs for medium/low defects)
# Reads: classification, defect_description
# Writes: recommendation
def node_standard_handler(state: DefectState) -> DefectState:
    print('[Node 3 - Standard Handler] Processing standard defect...')

    recommendation = standard_chain.invoke({
        'classification': state['classification'],
        'defect':         state['defect_description']
    })

    return {**state, 'recommendation': recommendation}


print('Node 2 and Node 3 defined')

Node 2 and Node 3 defined


In [10]:
# Node 4 — Report Writer (always runs last)
# Reads: defect_description, classification, recommendation
# Writes: final_report
def node_report_writer(state: DefectState) -> DefectState:
    print('[Node 4 - Report Writer] Writing final report...')

    report = report_chain.invoke({
        'defect':          state['defect_description'],
        'classification':  state['classification'],
        'recommendation':  state['recommendation']
    })

    return {**state, 'final_report': report}


# Routing function — reads severity_level and returns next node name
def route_by_severity(state: DefectState) -> str:
    print(f'[Router] Routing to: {state["severity_level"]} handler')
    if state['severity_level'] == 'critical':
        return 'critical_handler'   # must match the node name below
    else:
        return 'standard_handler'


print('Node 4 and routing function defined')

Node 4 and routing function defined


---
## Step 7 — Build the Graph

In [11]:
graph = StateGraph(DefectState)

# Add all nodes
graph.add_node('classifier',       node_classifier)
graph.add_node('critical_handler', node_critical_handler)
graph.add_node('standard_handler', node_standard_handler)
graph.add_node('report_writer',    node_report_writer)

# Entry point — always starts at classifier
graph.set_entry_point('classifier')

# After classifier — conditional route based on severity
graph.add_conditional_edges(
    'classifier',
    route_by_severity,
    {
        'critical_handler': 'critical_handler',
        'standard_handler': 'standard_handler'
    }
)

# Both handlers feed into the report writer
graph.add_edge('critical_handler', 'report_writer')
graph.add_edge('standard_handler', 'report_writer')

# Report writer always ends the graph
graph.add_edge('report_writer', END)

app = graph.compile()

print('Graph compiled successfully')
print()
print('Flow:')
print('  classifier')
print('     ├── [Critical/High] → critical_handler → report_writer → END')
print('     └── [Medium/Low]   → standard_handler → report_writer → END')

Graph compiled successfully

Flow:
  classifier
     ├── [Critical/High] → critical_handler → report_writer → END
     └── [Medium/Low]   → standard_handler → report_writer → END


---
## Step 8 — Run With a Critical Defect

In [12]:
critical_defect = """
DEF-2201: Production Issue — Fund Transfer
Environment: Production
Reported by: Customer Support

Description:
Customers are reporting that fund transfers above $5,000 are being processed twice.
The duplicate transaction appears in the account history within 2-3 minutes of the
original. Both transactions show status COMPLETED and both amounts are debited.
Approximately 47 customers affected since 09:00 AM today.
Total financial impact estimated at $340,000 in duplicate debits.
"""

initial_state: DefectState = {
    'defect_description': critical_defect.strip(),
    'classification':     '',
    'severity_level':     '',
    'recommendation':     '',
    'final_report':       ''
}

print('Running agent on CRITICAL defect...')
print('=' * 50)
critical_result = app.invoke(initial_state)
print('=' * 50)
print('Done')

Running agent on CRITICAL defect...
[Node 1 - Classifier] Classifying defect...
  Severity level determined: critical
[Router] Routing to: critical handler
[Node 2 - Critical Handler] Processing critical defect...
[Node 4 - Report Writer] Writing final report...
Done


In [13]:
print('CLASSIFICATION:')
print(critical_result['classification'])
print()
print('RECOMMENDATION:')
print(critical_result['recommendation'])
print()
print('FINAL REPORT:')
print(critical_result['final_report'])

CLASSIFICATION:
SEVERITY: Critical
TYPE: Functional
AFFECTED AREA: Fund Transfer
SUMMARY: Duplicate fund transfers above $5,000 are being processed twice, resulting in an estimated total financial impact of $340,000 in duplicate debits to approximately 47 customers.

RECOMMENDATION:
**Immediate Actions (Next 2 Hours)**

1. **Activate Emergency Patch**: Immediately activate a temporary patch to prevent further duplicate fund transfers. This patch should be applied ASAP to production environment.
2. **Notify Stakeholders**: Inform key stakeholders, including management, customer support, and IT teams, about the critical defect and its impact on customers.
3. **Escalate to Dev Team**: Immediately escalate this issue to the development team responsible for the Fund Transfer module to investigate the root cause and provide a fix as soon as possible.
4. **Activate Emergency Support Process**: Activate the emergency support process to ensure that customer support teams are aware of the situat

---
## Step 9 — Run With a Standard Defect

In [14]:
standard_defect = """
DEF-2198: UI Issue — Account Statement Page
Environment: UAT
Reported by: QA Team

Description:
On the account statement page, the date filter dropdown does not retain
the selected date range when the user navigates away and returns.
The filter resets to the default 30-day range every time.
Affects all browsers. No financial impact.
"""

standard_initial: DefectState = {
    'defect_description': standard_defect.strip(),
    'classification':     '',
    'severity_level':     '',
    'recommendation':     '',
    'final_report':       ''
}

print('Running agent on STANDARD defect...')
print('=' * 50)
standard_result = app.invoke(standard_initial)
print('=' * 50)
print('Done')

Running agent on STANDARD defect...
[Node 1 - Classifier] Classifying defect...
  Severity level determined: standard
[Router] Routing to: standard handler
[Node 3 - Standard Handler] Processing standard defect...
[Node 4 - Report Writer] Writing final report...
Done


In [15]:
print('CLASSIFICATION:')
print(standard_result['classification'])
print()
print('RECOMMENDATION:')
print(standard_result['recommendation'])
print()
print('FINAL REPORT:')
print(standard_result['final_report'])

CLASSIFICATION:
SEVERITY: Medium
TYPE: UI
AFFECTED AREA: Account Statement Page
SUMMARY: The account statement page's date filter dropdown does not retain the selected date range, resetting to the default 30-day range upon navigation away and return.

RECOMMENDATION:
**Suggested Fix Approach:**

1. Review the JavaScript code responsible for handling the date filter dropdown on the account statement page.
2. Identify the specific issue causing the selected date range to reset when navigating away and returning (e.g., is it related to session storage, local storage, or a specific event listener).
3. Implement a solution that persists the selected date range across page reloads, such as:
	* Using `localStorage` to store the selected date range.
	* Setting a cookie with the selected date range.
	* Utilizing a library like jQuery UI's `datepicker` plugin to handle date range selection and persistence.

**Test Cases to Verify the Fix:**

1. **Test Case 1:** Validate that the selected date ra

---
## Step 10 — Pretty Display

In [16]:
# Use Markdown display for a cleaner view in the notebook
for label, result in [('CRITICAL DEFECT', critical_result), ('STANDARD DEFECT', standard_result)]:
    report_md = (
        f'## {label}\n\n'
        f'### Classification\n{result["classification"]}\n\n'
        f'### Recommendation\n{result["recommendation"]}\n\n'
        f'### Manager Report\n{result["final_report"]}\n\n'
        f'---'
    )
    display(Markdown(report_md))

## CRITICAL DEFECT

### Classification
SEVERITY: Critical
TYPE: Functional
AFFECTED AREA: Fund Transfer
SUMMARY: Duplicate fund transfers above $5,000 are being processed twice, resulting in an estimated total financial impact of $340,000 in duplicate debits to approximately 47 customers.

### Recommendation
**Immediate Actions (Next 2 Hours)**

1. **Activate Emergency Patch**: Immediately activate a temporary patch to prevent further duplicate fund transfers. This patch should be applied ASAP to production environment.
2. **Notify Stakeholders**: Inform key stakeholders, including management, customer support, and IT teams, about the critical defect and its impact on customers.
3. **Escalate to Dev Team**: Immediately escalate this issue to the development team responsible for the Fund Transfer module to investigate the root cause and provide a fix as soon as possible.
4. **Activate Emergency Support Process**: Activate the emergency support process to ensure that customer support teams are aware of the situation and can provide guidance to affected customers.

**Likely Root Cause**

Based on the description, I believe the likely root cause is a misconfiguration or bug in the fund transfer processing logic. Specifically:

* A faulty condition check might be allowing duplicate transactions above $5,000 to pass through.
* An incorrect update of the transaction status might be causing both transactions to appear as completed.

**Regression Areas to Retest**

To ensure that the fix does not introduce new issues, I recommend retesting the following areas:

1. **Fund Transfer Module**: Re-test the entire fund transfer module to ensure that no duplicate transactions are processed.
2. **Account History**: Verify that account history is updated correctly and that duplicate transactions do not appear in the account history.
3. **Transaction Status**: Confirm that transaction status updates are correct and that both transactions show as completed.

**Rollback Recommendation**

Yes, I recommend rolling back to a stable previous version of the codebase. The financial impact of $340,000 is significant, and it's crucial to minimize further damage until a fix can be implemented. Rolling back will prevent any additional duplicate fund transfers from occurring, which could exacerbate the issue.

Additionally, since this is a critical defect in production, I would recommend implementing a rollback procedure as soon as possible to ensure that we can quickly revert to a stable state and minimize the financial impact on customers.

### Manager Report
Here's a rewritten version of the defect analysis report:

**DEF-2201: Production Issue - Fund Transfer**

**Severity:** Critical
**Impact:** Estimated total financial impact of $340,000 in duplicate debits to approximately 47 customers.
**Type:** Functional
**Affecting Area:** Fund Transfer

**Summary:**
Duplicate fund transfers above $5,000 are being processed twice, resulting in an estimated total financial impact. The issue was reported by customer support and affects approximately 47 customers since 09:00 AM today.

**Recommendations:**

1. **Immediate Actions (Next 2 Hours)**:
	* Activate a temporary patch to prevent further duplicate fund transfers.
	* Notify key stakeholders, including management, customer support, and IT teams.
	* Escalate the issue to the development team responsible for the Fund Transfer module to investigate the root cause and provide a fix as soon as possible.
	* Activate the emergency support process to ensure that customer support teams are aware of the situation.
2. **Likely Root Cause**: Misconfiguration or bug in the fund transfer processing logic, possibly due to a faulty condition check allowing duplicate transactions above $5,000 to pass through.

**Regression Areas to Retest:**

1. Fund Transfer Module
2. Account History
3. Transaction Status

**Rollback Recommendation:** Yes, I recommend rolling back to a stable previous version of the codebase to minimize further damage and prevent additional duplicate fund transfers from occurring.

This rewritten report maintains the same level of detail and professionalism as the original while condensing the information into a more concise format.

---

## STANDARD DEFECT

### Classification
SEVERITY: Medium
TYPE: UI
AFFECTED AREA: Account Statement Page
SUMMARY: The account statement page's date filter dropdown does not retain the selected date range, resetting to the default 30-day range upon navigation away and return.

### Recommendation
**Suggested Fix Approach:**

1. Review the JavaScript code responsible for handling the date filter dropdown on the account statement page.
2. Identify the specific issue causing the selected date range to reset when navigating away and returning (e.g., is it related to session storage, local storage, or a specific event listener).
3. Implement a solution that persists the selected date range across page reloads, such as:
	* Using `localStorage` to store the selected date range.
	* Setting a cookie with the selected date range.
	* Utilizing a library like jQuery UI's `datepicker` plugin to handle date range selection and persistence.

**Test Cases to Verify the Fix:**

1. **Test Case 1:** Validate that the selected date range is retained when navigating away from the account statement page and returning.
	* Preconditions: Select a valid date range on the account statement page.
	* Steps:
		+ Navigate away from the page (e.g., click the back button).
		+ Return to the page.
		+ Verify that the selected date range is still displayed.
2. **Test Case 2:** Validate that the default 30-day range is not applied when navigating away and returning.
	* Preconditions: Select a different date range on the account statement page (e.g., 60 days).
	* Steps:
		+ Navigate away from the page (e.g., click the back button).
		+ Return to the page.
		+ Verify that the selected date range is still displayed, not the default 30-day range.
3. **Test Case 3:** Validate that the fix works across different browsers.
	* Preconditions: Select a valid date range on the account statement page in multiple browsers (e.g., Chrome, Firefox, Safari).
	* Steps:
		+ Navigate away from each browser instance and return to verify that the selected date range is retained.

**Related Areas to do a Quick Sanity Check On:**

1. **Session Management:** Review how sessions are handled on the account statement page to ensure that the fix doesn't introduce any session-related issues.
2. **Cookie Management:** Verify that cookies are not being set or updated unnecessarily, which could affect the persistence of the selected date range.
3. **Browser Compatibility:** Quickly review browser-specific features and plugins used on the account statement page to ensure compatibility with the suggested fix.

By following these steps, you should be able to identify and fix the issue causing the date filter dropdown to reset when navigating away and returning.

### Manager Report
Here is a rewritten version of the defect analysis report in one paragraph:

**Defect Analysis Report: DEF-2198**

The account statement page's date filter dropdown does not retain the selected date range, resetting to the default 30-day range upon navigation away and return. This issue affects all browsers with no financial impact. The severity is classified as Medium due to its UI-related nature. To fix this defect, we recommend reviewing the JavaScript code responsible for handling the date filter dropdown and implementing a solution that persists the selected date range across page reloads using `localStorage`, setting a cookie, or utilizing a library like jQuery UI's `datepicker` plugin. We will test the fix with three scenarios: (1) verifying that the selected date range is retained when navigating away from the page and returning, (2) confirming that the default 30-day range is not applied in this scenario, and (3) ensuring compatibility across multiple browsers. Additionally, we will perform quick sanity checks on session management, cookie management, and browser-specific features to ensure a smooth implementation.

---

---
## What You Just Built — End to End

```
Defect Input
     ↓
[Node 1: Classifier]     ← LangChain chain inside (prompt | llm | parser)
     ↓
[Router Function]        ← reads severity_level from State
     ├── Critical → [Node 2: Critical Handler] ← different chain
     └── Standard → [Node 3: Standard Handler] ← different chain
                           ↓
                   [Node 4: Report Writer]      ← always runs
                           ↓
                          END
```

| Layer | Tool | Job |
|---|---|---|
| LLM | Ollama / llama3.2 | Generates the text |
| Prompt Template | LangChain | Structures the instruction |
| Chain (pipe) | LangChain | Connects prompt → llm → parser |
| State | LangGraph | Shared notepad between nodes |
| Node | LangGraph | One function, one job, reads/writes state |
| Conditional Edge | LangGraph | Decides which node runs next |
| Graph | LangGraph | The entire workflow |

**You now have the full foundation to understand and extend the 3 QA agent notebooks.**